In [ ]:
import copy
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset


# =============================================================
# 0. CHUẨN BỊ MÔI TRƯỜNG & SEED (CÓ RESET CON TRỎ DATALOADER GENERATOR)
# =============================================================
def set_seed(seed=42, g=None):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if g is not None:
        g.manual_seed(seed)  # Reset con trỏ DataLoader Generator về seed 42


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang thực thi trên thiết bị: {device}")

os.makedirs("./plots", exist_ok=True)


# =============================================================
# 1. FOCAL LOSS DYNAMIC
# =============================================================
class FocalLoss(nn.Module):

    def __init__(self, alpha=0.50, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction="none"
        )
        pt = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean() if self.reduction == "mean" else focal


# =============================================================
# 2. DATASET CHUẨN (102 FEATURES)
# =============================================================
class OptimizedSkeletonDataset(Dataset):

    def __init__(self, sequences, labels, max_len=16, is_train=False):
        self.is_train = is_train
        self.max_len = max_len
        self.labels = np.array(labels, dtype=np.float32)
        self.sequences = []

        for seq in sequences:
            seq = np.array(seq, dtype=np.float32)
            raw_flat = seq.reshape(seq.shape[0], -1)

            T, _ = raw_flat.shape
            if T >= max_len:
                padded_raw = raw_flat[:max_len]
            else:
                padding = np.tile(raw_flat[0], (max_len - T, 1))
                padded_raw = np.vstack((padding, raw_flat))

            velocity = np.diff(
                padded_raw, axis=0, append=padded_raw[-1:]
            )
            acceleration = np.diff(velocity, axis=0, append=velocity[-1:])
            enriched = np.hstack((padded_raw, velocity, acceleration))

            self.sequences.append(enriched.astype(np.float32))

        self.sequences = np.array(self.sequences, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx].copy()
        if self.is_train and self.labels[idx] == 1 and random.random() < 0.5:
            seq = seq + np.random.normal(0, 0.02, seq.shape).astype(np.float32)
        return torch.tensor(seq), torch.tensor(self.labels[idx])


# =============================================================
# 3. ĐỊNH NGHĨA 4 KIẾN TRÚC MÔ HÌNH
# =============================================================
class BiGRUAttentionModel(nn.Module):

    def __init__(
        self, input_size=102, hidden_size=64, num_layers=2, dropout_rate=0.3
    ):
        super().__init__()
        # Đổi self.gru -> self.rnn để trùng thuộc tính với UnifiedAblationModel
        self.rnn = nn.GRU(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.dropout_layer = nn.Dropout(p=dropout_rate)
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.dropout_layer(out)
        attn_scores = self.attention(out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(out * attn_weights, dim=1)
        return self.fc(context)


class LSTMModel(nn.Module):

    def __init__(
        self, input_size=102, hidden_size=64, num_layers=2, dropout_rate=0.3
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class GRUModel(nn.Module):

    def __init__(
        self, input_size=102, hidden_size=64, num_layers=2, dropout_rate=0.3
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


class CNN1DModel(nn.Module):

    def __init__(self, input_size=102, hidden_size=64, dropout_rate=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(
            in_channels=input_size,
            out_channels=hidden_size,
            kernel_size=3,
            padding=1,
        )
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = torch.flatten(self.pool(x), 1)
        return self.fc(x)


# =============================================================
# 4. HÀM ĐÁNH GIÁ
# =============================================================
def evaluate_model(model, dataloader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for seqs, labels in dataloader:
            seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).float()
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return acc, rec, prec, f1


# =============================================================
# 5. KHẢO SÁT MA TRẬN ALPHA x GAMMA CHO CÁC MÔ HÌNH
# =============================================================
def run_grid_matrix_experiment(
    train_loader, val_loader, models_dict, alphas, gammas, g=None, epochs=20
):
    results_list = []

    for model_name, model_class in models_dict.items():
        print("\n=======================================================")
        print(f"  ĐANG CHẠY MA TRẬN THAM SỐ CHO MÔ HÌNH: [{model_name}]")
        print("=======================================================")

        for g_val in gammas:
            for a_val in alphas:
                # RESET SEED VÀ GENERATOR g TRƯỚC MỖI VÒNG LẶP ĐỂ ĐỒNG BỘ 100%
                set_seed(42, g)

                model = model_class().to(device)
                criterion = FocalLoss(alpha=a_val, gamma=g_val)
                optimizer = torch.optim.AdamW(
                    model.parameters(), lr=0.001, weight_decay=0.01
                )

                best_val_f1 = -1.0
                best_val_rec = -1.0

                for epoch in range(epochs):
                    model.train()
                    for seqs, labels in train_loader:
                        seqs, labels = seqs.to(device), labels.to(
                            device
                        ).unsqueeze(1)
                        optimizer.zero_grad()
                        criterion(model(seqs), labels).backward()
                        optimizer.step()

                    acc, rec, prec, f1 = evaluate_model(
                        model, val_loader, criterion
                    )
                    if f1 > best_val_f1:
                        best_val_f1 = f1
                        best_val_rec = rec

                print(
                    f"  Model: {model_name:<16} | Gamma={g_val:<3} | Alpha={a_val:<4} ==> Val F1: {best_val_f1*100:5.2f}% | Val Recall: {best_val_rec*100:5.2f}%"
                )

                results_list.append({
                    "Model": model_name,
                    "Gamma": g_val,
                    "Alpha": a_val,
                    "F1-Score": round(best_val_f1 * 100, 2),
                    "Recall": round(best_val_rec * 100, 2),
                })

    return pd.DataFrame(results_list)


# =============================================================
# 6. MAIN PIPELINE EXECUTION
# =============================================================
def main():
    X_path = "./data_processed/imvia_sequences.npy"
    y_path = "./data_processed/imvia_labels.npy"

    if not os.path.exists(X_path) or not os.path.exists(y_path):
        X_path = "./data_processed/sequences.npy"
        y_path = "./data_processed/labels.npy"

    X = np.load(X_path, allow_pickle=True)
    y = np.load(y_path)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.176,
        random_state=42,
        stratify=y_train_val,
    )

    def seed_worker(worker_id):
        np.random.seed(42 + worker_id)
        random.seed(42 + worker_id)

    g = torch.Generator().manual_seed(42)
    set_seed(42, g)

    train_loader = DataLoader(
        OptimizedSkeletonDataset(X_train, y_train, max_len=16, is_train=True),
        batch_size=16,
        shuffle=True,
        worker_init_fn=seed_worker,
        generator=g,
    )
    val_loader = DataLoader(
        OptimizedSkeletonDataset(X_val, y_val, max_len=16),
        batch_size=16,
        shuffle=False,
    )

    # 1. Định nghĩa các mô hình thử nghiệm
    models_dict = {
        "BiGRU-Attention": BiGRUAttentionModel,
        "LSTM": LSTMModel,
        "GRU": GRUModel,
        "CNN-1D": CNN1DModel,
    }

    # 2. Khai báo dải Gamma và Alpha
    alphas = [0.1, 0.25, 0.5, 0.75, 0.9]
    gammas = [0.1, 0.2, 0.5, 1.0, 2.0]

    # 3. Chạy thực nghiệm (Truyền đối tượng g vào)
    df_results = run_grid_matrix_experiment(
        train_loader, val_loader, models_dict, alphas, gammas, g=g, epochs=20
    )

    # 4. IN BẢNG MA TRẬN PIVOT CHO TỪNG MÔ HÌNH
    print(
        "\n========================================================================="
    )
    print("      IN BẢNG MA TRẬN PIVOT F1-SCORE (%) (GAMMA x ALPHA)")
    print(
        "========================================================================="
    )

    for model_name in models_dict.keys():
        df_sub = df_results[df_results["Model"] == model_name]
        pivot_f1 = df_sub.pivot(
            index="Gamma", columns="Alpha", values="F1-Score"
        )

        print(f"\n---> MA TRẬN F1-SCORE FOR MODEL: [{model_name}]")
        print(pivot_f1)
        print("-" * 50)

    # 5. TRỰC QUAN HÓA BẰNG 4 HEATMAPS SIDE-BY-SIDE
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, model_name in enumerate(models_dict.keys()):
        df_sub = df_results[df_results["Model"] == model_name]
        pivot_f1 = df_sub.pivot(
            index="Gamma", columns="Alpha", values="F1-Score"
        )

        sns.heatmap(
            pivot_f1,
            annot=True,
            fmt=".1f",
            cmap="YlGnBu",
            ax=axes[idx],
            cbar=True,
        )
        axes[idx].set_title(
            f"Model: {model_name} (F1-Score %)", fontweight="bold"
        )
        axes[idx].set_xlabel("Alpha (α)")
        axes[idx].set_ylabel("Gamma (γ)")

    plt.tight_layout()
    plt.savefig("./plots/grid_matrix_4models_heatmap.png", dpi=300)
    plt.close()

    print(
        "\n[OK] Đã xuất biểu đồ 4 Heatmaps tại: './plots/grid_matrix_4models_heatmap.png'"
    )


if __name__ == "__main__":
    main()

In [5]:
import copy
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

# Cài đặt tự động hoặc kiểm tra thư viện 'thop' để tính FLOPS & Parameters
try:
    from thop import profile
except ImportError:
    print("Đang cài đặt thư viện 'thop' để đo FLOPS...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "thop", "-q"])
    from thop import profile


# =============================================================
# 0. CHUẨN BỊ MÔI TRƯỜNG & SEED
# =============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang thực thi Lần chạy 2 (Ablation Study) trên: {device}")

os.makedirs("./plots", exist_ok=True)


# =============================================================
# HÀM TÍNH KÍCH THƯỚC MODEL (PARAMS) VÀ FLOPS
# =============================================================
def get_model_complexity(model, input_size=(1, 16, 102)):
    """
    Tính tổng tham số (Parameters) và số phép tính điểm động (FLOPs / MACs)
    Input shape mặc định: batch_size=1, sequence_length=16, features=102 (hoặc 34)
    """
    model.eval()
    dummy_input = torch.randn(input_size).to(device)

    # Tính số lượng tham số học được (Trainable Parameters)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Tính MACs / FLOPs thông qua thư viện thop
    try:
        macs, _ = profile(model, inputs=(dummy_input,), verbose=False)
        flops = macs * 2  # 1 MAC ≈ 2 FLOPs
    except Exception:
        flops = 0.0

    return total_params, flops


# =============================================================
# 1. FOCAL LOSS DYNAMIC & BCE WITH LOGITS
# =============================================================
class FocalLoss(nn.Module):

    def __init__(self, alpha=0.5, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction="none"
        )
        pt = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean() if self.reduction == "mean" else focal


# =============================================================
# 2. DATASET CÓ THỂ BẬT/TẮT KINEMATICS (102 DIMS VS 34 DIMS)
# =============================================================
class FlexibleSkeletonDataset(Dataset):

    def __init__(
        self,
        sequences,
        labels,
        max_len=16,
        use_kinematics=True,
        is_train=False,
    ):
        self.is_train = is_train
        self.labels = np.array(labels, dtype=np.float32)
        self.sequences = []

        for seq in sequences:
            seq = np.array(seq, dtype=np.float32)
            raw_flat = seq.reshape(seq.shape[0], -1)

            T, _ = raw_flat.shape
            if T >= max_len:
                padded_raw = raw_flat[:max_len]
            else:
                padding = np.tile(raw_flat[0], (max_len - T, 1))
                padded_raw = np.vstack((padding, raw_flat))

            if use_kinematics:
                velocity = np.diff(
                    padded_raw, axis=0, append=padded_raw[-1:]
                )
                acceleration = np.diff(velocity, axis=0, append=velocity[-1:])
                enriched = np.hstack((padded_raw, velocity, acceleration))  # 102
            else:
                enriched = padded_raw  # 34 (Pose thô)

            self.sequences.append(enriched.astype(np.float32))

        self.sequences = np.array(self.sequences, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx].copy()
        if self.is_train and self.labels[idx] == 1 and random.random() < 0.5:
            seq = seq + np.random.normal(0, 0.02, seq.shape).astype(np.float32)
        return torch.tensor(seq), torch.tensor(self.labels[idx])


# =============================================================
# 3. MÔ HÌNH HỖ TRỢ BẬT/TẮT ATTENTION & CHUYỂN ĐỔI BiGRU/LSTM
# =============================================================
class UnifiedAblationModel(nn.Module):

    def __init__(
        self,
        input_size=102,
        hidden_size=64,
        num_layers=2,
        rnn_type="GRU",
        use_attention=True,
        bidirectional=True,
        dropout_rate=0.3,
    ):
        super().__init__()
        self.use_attention = use_attention
        RNNClass = nn.GRU if rnn_type == "GRU" else nn.LSTM

        self.rnn = RNNClass(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.dropout_layer = nn.Dropout(p=dropout_rate)

        if use_attention:
            self.attention = nn.Linear(out_dim, 1)
        self.fc = nn.Linear(out_dim, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.dropout_layer(out)

        if self.use_attention:
            attn_scores = self.attention(out)
            attn_weights = torch.softmax(attn_scores, dim=1)
            context = torch.sum(out * attn_weights, dim=1)
            return self.fc(context)
        else:
            # Nếu bỏ Attention: Lấy hidden state ở bước thời gian cuối cùng
            return self.fc(out[:, -1, :])


# =============================================================
# 4. HÀM ĐÁNH GIÁ & TRAIN
# =============================================================
def evaluate_model(model, dataloader, criterion, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for seqs, labels in dataloader:
            seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(seqs)
            preds = (torch.sigmoid(outputs) >= threshold).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return acc, rec, prec, f1


def train_ablation_candidate(
    model, train_loader, val_loader, criterion, epochs=20
):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=0.001, weight_decay=0.01
    )
    best_f1 = -1.0
    best_metrics = ()

    for epoch in range(epochs):
        model.train()
        for seqs, labels in train_loader:
            seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(seqs), labels)
            loss.backward()
            optimizer.step()

        acc, rec, prec, f1 = evaluate_model(model, val_loader, criterion)
        if f1 > best_f1:
            best_f1 = f1
            best_metrics = (acc, rec, prec, f1)

    return best_metrics


# =============================================================
# 5. CHẠY CÁC KỊCH BẢN CẮT BỎ (ABLATION EXPERIMENTS)
# =============================================================
def run_ablation_pipeline(
    train_full, val_full, train_pose, val_pose, epochs=20
):
    print("\n=========================================================================")
    print("      BẮT ĐẦU CHẠY CÁC KỊCH BẢN ABLATION STUDY (NGHIÊN CỨU CẮT BỎ)")
    print("=========================================================================")

    ablation_experiments = {
        "1. PROPOSED (BiGRU + Attention + Kinematics)": {
            "rnn": "GRU",
            "attn": True,
            "kinematics": True,
            "bce": False,
        },
        "2. BiGRU (Bỏ Attention)": {
            "rnn": "GRU",
            "attn": False,
            "kinematics": True,
            "bce": False,
        },
        "3. LSTM + Attention": {
            "rnn": "LSTM",
            "attn": True,
            "kinematics": True,
            "bce": False,
        },
        "4. Vanilla LSTM (Bỏ Attention)": {
            "rnn": "LSTM",
            "attn": False,
            "kinematics": True,
            "bce": False,
        },
        "5. Bỏ Kinematics (Chỉ Pose thô 34 dims)": {
            "rnn": "GRU",
            "attn": True,
            "kinematics": False,
            "bce": False,
        },
        "6. Bỏ Focal Loss (Standard BCE)": {
            "rnn": "GRU",
            "attn": True,
            "kinematics": True,
            "bce": True,
        },
    }

    focal_criterion = FocalLoss(alpha=0.5, gamma=2.0)
    bce_criterion = nn.BCEWithLogitsLoss()

    ablation_results = []

    for name, cfg in ablation_experiments.items():
        set_seed(42)
        use_kin = cfg["kinematics"]
        t_loader = train_full if use_kin else train_pose
        v_loader = val_full if use_kin else val_pose
        in_dim = 102 if use_kin else 34

        model = UnifiedAblationModel(
            input_size=in_dim,
            hidden_size=64,
            num_layers=2,
            rnn_type=cfg["rnn"],
            use_attention=cfg["attn"],
            bidirectional=True,
            dropout_rate=0.3,
        ).to(device)

        # Tính Parameters và FLOPs cho biến thể hiện tại
        params, flops = get_model_complexity(model, input_size=(1, 16, in_dim))

        crit = bce_criterion if cfg["bce"] else focal_criterion
        acc, rec, prec, f1 = train_ablation_candidate(
            model, t_loader, v_loader, crit, epochs=epochs
        )

        flops_m = flops / 1e6
        print(
            f"  --> {name:<46} | F1: {f1*100:5.2f}% | Recall: {rec*100:5.2f}% | Params: {params:,} | FLOPs: {flops_m:.4f}M"
        )

        ablation_results.append({
            "Biến thể Cấu hình": name,
            "Accuracy (%)": round(acc * 100, 2),
            "Precision (%)": round(prec * 100, 2),
            "Recall (%)": round(rec * 100, 2),
            "F1-Score (%)": round(f1 * 100, 2),
            "Parameters": params,
            "FLOPs (M)": round(flops_m, 4),
        })

    return pd.DataFrame(ablation_results)


# =============================================================
# 6. MAIN EXECUTION
# =============================================================
def main():
    X_path = "./data_processed/imvia_sequences.npy"
    y_path = "./data_processed/imvia_labels.npy"

    if not os.path.exists(X_path) or not os.path.exists(y_path):
        X_path = "./data_processed/sequences.npy"
        y_path = "./data_processed/labels.npy"

    X = np.load(X_path, allow_pickle=True)
    y = np.load(y_path)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.176,
        random_state=42,
        stratify=y_train_val,
    )

    def seed_worker(worker_id):
        np.random.seed(42 + worker_id)
        random.seed(42 + worker_id)

    g = torch.Generator().manual_seed(42)

    # Dữ liệu đầy đủ (102 đặc trưng)
    train_full = DataLoader(
        FlexibleSkeletonDataset(
            X_train, y_train, use_kinematics=True, is_train=True
        ),
        batch_size=16,
        shuffle=True,
        worker_init_fn=seed_worker,
        generator=g,
    )
    val_full = DataLoader(
        FlexibleSkeletonDataset(
            X_val, y_val, use_kinematics=True, is_train=False
        ),
        batch_size=16,
        shuffle=False,
    )

    # Dữ liệu cắt bỏ Kinematics (Chỉ Pose thô 34 đặc trưng)
    train_pose = DataLoader(
        FlexibleSkeletonDataset(
            X_train, y_train, use_kinematics=False, is_train=True
        ),
        batch_size=16,
        shuffle=True,
        worker_init_fn=seed_worker,
        generator=g,
    )
    val_pose = DataLoader(
        FlexibleSkeletonDataset(
            X_val, y_val, use_kinematics=False, is_train=False
        ),
        batch_size=16,
        shuffle=False,
    )

    # Chạy Ablation Study
    df_ablation = run_ablation_pipeline(
        train_full, val_full, train_pose, val_pose, epochs=20
    )

    # IN BẢNG BÁO CÁO ABLATION STUDY
    print(
        "\n========================================================================================================="
    )
    print("                                BẢNG KẾT QUẢ ABLATION STUDY HOÀN CHỈNH")
    print(
        "========================================================================================================="
    )
    print(df_ablation.to_string(index=False))

    # TRỰC QUAN HÓA BẰNG BIỂU ĐỒ SONG SONG (Dual-axis Chart)
    fig, ax1 = plt.subplots(figsize=(12, 6))

    x = np.arange(len(df_ablation["Biến thể Cấu hình"]))
    width = 0.35

    # Cột hiển thị F1-Score (Trục trái)
    color1 = "#1f77b4"
    ax1.set_ylabel("Phần trăm (%)", fontsize=11, color=color1)
    bars1 = ax1.bar(
        x - width / 2,
        df_ablation["F1-Score (%)"],
        width,
        label="F1-Score (%)",
        color=color1,
        alpha=0.8,
    )
    bars2 = ax1.bar(
        x + width / 2,
        df_ablation["Recall (%)"],
        width,
        label="Recall (%)",
        color="#ff7f0e",
        alpha=0.8,
    )
    ax1.set_ylim(50, 105)
    ax1.tick_params(axis='y', labelcolor=color1)

    # Đường hiển thị FLOPs (Trục phải)
    ax2 = ax1.twinx()
    color2 = "#d62728"
    ax2.set_ylabel("FLOPs (M)", fontsize=11, color=color2)
    lines = ax2.plot(
        x,
        df_ablation["FLOPs (M)"],
        color=color2,
        marker="o",
        linewidth=2.5,
        label="FLOPs (M)",
    )
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title(
        "Ablation Study: Danh gia vai tro cua tung thanh phan ky thuat va Do phuc tap tinh toan",
        fontweight="bold",
        fontsize=12,
    )
    short_labels = [
        name.split("(")[0].strip() for name in df_ablation["Biến thể Cấu hình"]
    ]
    ax1.set_xticks(x)
    ax1.set_xticklabels(short_labels, rotation=20, ha="right", fontsize=9)

    # Gộp legend từ 2 trục
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

    ax1.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("./plots/ablation_study_chart.png", dpi=300)
    plt.close()

    print(
        "\n[OK] Đã lưu biểu đồ Ablation Study tại: './plots/ablation_study_chart.png'"
    )


if __name__ == "__main__":
    main()

Đang thực thi Lần chạy 2 (Ablation Study) trên: cuda

      BẮT ĐẦU CHẠY CÁC KỊCH BẢN ABLATION STUDY (NGHIÊN CỨU CẮT BỎ)
  --> 1. PROPOSED (BiGRU + Attention + Kinematics)   | F1: 87.59% | Recall: 83.42% | Params: 139,266 | FLOPs: 4.5100M
  --> 2. BiGRU (Bỏ Attention)                        | F1: 85.22% | Recall: 80.71% | Params: 139,137 | FLOPs: 4.5059M
  --> 3. LSTM + Attention                            | F1: 85.75% | Recall: 82.61% | Params: 185,602 | FLOPs: 6.0009M
  --> 4. Vanilla LSTM (Bỏ Attention)                 | F1: 84.33% | Recall: 80.43% | Params: 185,473 | FLOPs: 5.9968M
  --> 5. Bỏ Kinematics (Chỉ Pose thô 34 dims)        | F1: 86.89% | Recall: 82.88% | Params: 113,154 | FLOPs: 3.6744M
  --> 6. Bỏ Focal Loss (Standard BCE)                | F1: 86.70% | Recall: 82.34% | Params: 139,266 | FLOPs: 4.5100M

                                BẢNG KẾT QUẢ ABLATION STUDY HOÀN CHỈNH
                           Biến thể Cấu hình  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)

In [2]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score, accuracy_score, f1_score,
    confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Đảm bảo tính tái lập
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng cấu hình: {device}")

# =============================================================
# 1. FOCAL LOSS CHUẨN
# =============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.50, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean() if self.reduction == 'mean' else focal

# =============================================================
# 2. DATASET
# =============================================================
class OptimizedSkeletonDataset(Dataset):
    def __init__(self, sequences, labels, max_len=16, is_train=False):
        self.is_train = is_train
        self.max_len = max_len
        self.labels = np.array(labels, dtype=np.float32)
        self.sequences = []

        for seq in sequences:
            seq = np.array(seq, dtype=np.float32)
            raw_flat = seq.reshape(seq.shape[0], -1)

            T, F = raw_flat.shape
            if T >= max_len:
                padded_raw = raw_flat[:max_len]
            else:
                padding = np.tile(raw_flat[0], (max_len - T, 1))
                padded_raw = np.vstack((padding, raw_flat))

            velocity = np.diff(padded_raw, axis=0, append=padded_raw[-1:]) # Tính sự chênh lệch tọa độ giữa 2 frame liên tiếp
            acceleration = np.diff(velocity, axis=0, append=velocity[-1:])
            enriched = np.hstack((padded_raw, velocity, acceleration)) # Ghép 3 ma trận lại theo chiều ngang.

            self.sequences.append(enriched.astype(np.float32))

        self.sequences = np.array(self.sequences, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx].copy()
        if self.is_train and self.labels[idx] == 1 and random.random() < 0.5:
            seq = seq + np.random.normal(0, 0.02, seq.shape).astype(np.float32)
        return torch.tensor(seq), torch.tensor(self.labels[idx])

# =============================================================
# 3. MÔ HÌNH BiGRU-ATTENTION
# =============================================================
class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_size=102, hidden_size=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        attn_scores = self.attention(out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(out * attn_weights, dim=1)
        return self.fc(context)

# =============================================================
# 4. ĐÁNH GIÁ
# =============================================================
def evaluate_model(model, dataloader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for seqs, labels in dataloader:
            seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).float()
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    avg_loss = total_loss / len(dataloader)

    return avg_loss, acc, rec, f1, all_labels, all_preds, all_probs

def find_optimal_threshold(all_labels, all_probs):
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    idx = np.argmax(tpr - fpr)
    return float(thresholds[idx])

# =============================================================
# 5. GRID SEARCH THEO CHỈ SỐ F1-SCORE LÀM GỐC
# =============================================================
def run_grid_search(train_loader, val_loader, criterion, grid_space, probe_epochs=35):
    best_f1     = -1.0
    best_recall = -1.0
    best_config = None

    print("\n==== GIAI ĐOẠN 1: GRID SEARCH (BiGRU-Attention) ====")
    for h_size in grid_space['hidden_size']:
        for n_layers in grid_space['num_layers']:
            for lr_rate in grid_space['lr']:
                config = {'hidden_size': h_size, 'num_layers': n_layers, 'lr': lr_rate}
                set_seed(42)
                model = BiGRUAttentionModel(hidden_size=h_size, num_layers=n_layers).to(device)
                optimizer = torch.optim.AdamW(model.parameters(), lr=lr_rate, weight_decay=0.01)

                best_probe_f1 = -1.0
                best_probe_state = None
                no_improve = 0
                patience = 7

                for epoch in range(probe_epochs):
                    model.train()
                    for seqs, labels in train_loader:
                        seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
                        optimizer.zero_grad()
                        criterion(model(seqs), labels).backward()
                        optimizer.step()

                    _, _, val_rec, val_f1, _, _, _ = evaluate_model(model, val_loader, criterion)

                    # Tối ưu trực tiếp theo F1-Score ở bước probe ngắn
                    if val_f1 > best_probe_f1:
                        best_probe_f1 = val_f1
                        best_probe_state = copy.deepcopy(model.state_dict())
                        no_improve = 0
                    else:
                        no_improve += 1
                    if no_improve >= patience:
                        break

                if best_probe_state is not None:
                    model.load_state_dict(best_probe_state)

                _, _, val_rec, val_f1, _, _, _ = evaluate_model(model, val_loader, criterion)
                print(f"  H={h_size} L={n_layers} LR={lr_rate} "
                      f"| Val F1: {val_f1*100:.1f}% | Val Recall: {val_rec*100:.1f}%")

                # Quyết định chọn cấu hình dựa trên F1-Score trước, Recall sau
                if val_f1 > best_f1 or (val_f1 == best_f1 and val_rec > best_recall):
                    best_f1     = val_f1
                    best_recall = val_rec
                    best_config = config

    print(f"\n[*] Config tối ưu chọn theo F1: {best_config}")
    return best_config

# =============================================================
# 6. HUẤN LUYỆN CHI TIẾT VÀ CHỐNG OVERFITTING THEO F1
# =============================================================
def train_final_model(train_loader, val_loader, best_config, criterion, epochs=50):
    model = BiGRUAttentionModel(
        hidden_size=best_config['hidden_size'],
        num_layers=best_config['num_layers']
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=best_config['lr'], weight_decay=0.01
    ) # Weight Decay (0.01): Giúp phạt các trọng số quá lớn trong mạng, hạn chế mô hình học thuộc lòng dữ liệu train.

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    best_f1_score   = -1.0
    best_model_state = None
    patience        = 10  # Siết chặt lại patience để chặn đứng Overfitting sớm hơn
    no_improve      = 0

    print("\n==== GIAI ĐOẠN 2: HUẤN LUYỆN CHI TIẾT (BiGRU-Attention) ====")

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for seqs, labels in train_loader:
            seqs, labels = seqs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item()
            preds          = (torch.sigmoid(outputs) >= 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc  = train_correct / train_total

        epoch_val_loss, epoch_val_acc, epoch_val_rec, epoch_val_f1, _, _, _ = \
            evaluate_model(model, val_loader, criterion)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        # Theo dõi Scheduler dựa trên Val F1 thực tế
        scheduler.step(epoch_val_f1)

        print(f"Epoch [{epoch+1:02d}/{epochs}] "
              f"TrainLoss: {epoch_train_loss:.4f} TrainAcc: {epoch_train_acc*100:.1f}% | "
              f"ValLoss: {epoch_val_loss:.4f} ValAcc: {epoch_val_acc*100:.1f}% "
              f"ValF1: {epoch_val_f1*100:.1f}% ValRecall: {epoch_val_rec*100:.1f}%")

        # Early Stopping lưu model tại đỉnh cao nhất của F1-Score
        if epoch_val_f1 > best_f1_score:
            best_f1_score    = epoch_val_f1
            best_model_state = copy.deepcopy(model.state_dict())
            no_improve       = 0
            print(f"  ==> Cải thiện! Toàn diện Val F1={best_f1_score*100:.1f}% — đã lưu trạng thái model.")
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"  [Early Stop] Không cải thiện F1 sau {patience} epoch. Ngắt để chống Overfit.")
            break

    if best_model_state is None:
        best_model_state = copy.deepcopy(model.state_dict())

    return model, best_model_state, history

# =============================================================
# 7. MAIN PIPELINE
# =============================================================
def main():
    X = np.load('./data_processed/imvia_sequences.npy', allow_pickle=True)
    y = np.load('./data_processed/imvia_labels.npy')

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
    )

    def seed_worker(worker_id):
        np.random.seed(42 + worker_id)
        random.seed(42 + worker_id)

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        OptimizedSkeletonDataset(X_train, y_train, max_len=16, is_train=True),
        batch_size=8, shuffle=True,
        worker_init_fn=seed_worker, generator=g
    )
    val_loader = DataLoader(
        OptimizedSkeletonDataset(X_val, y_val, max_len=16),
        batch_size=8, shuffle=False
    )
    test_loader = DataLoader(
        OptimizedSkeletonDataset(X_test, y_test, max_len=16),
        batch_size=8, shuffle=False
    )

    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    grid_space = {
        'hidden_size': [32, 64],
        'num_layers':  [1, 2],
        'lr':          [0.001, 0.005]
    }

    # Tìm cấu hình tốt nhất ở Giai đoạn 1
    best_config = run_grid_search(train_loader, val_loader, criterion, grid_space, probe_epochs=35)

    # Train mô hình hoàn chỉnh ở Giai đoạn 2
    final_model, best_model_state, history = train_final_model(
        train_loader, val_loader, best_config, criterion, epochs=50
    )

    final_model.load_state_dict(best_model_state) # Nạp lại bộ trọng số tốt nhất đã lưu.
    # Chạy dự đoán trên tập Validation để tìm optimal_threshold qua Youden Index
    _, _, _, _, val_true, _, val_probs = evaluate_model(final_model, val_loader, criterion)
    optimal_threshold = find_optimal_threshold(val_true, val_probs)
    print(f"\n[*] Threshold tối ưu (Youden Index trên Val): {optimal_threshold:.4f}")

    test_loss, test_acc, test_rec, test_f1, true_labels, pred_labels, _ = \
        evaluate_model(final_model, test_loader, criterion, threshold=optimal_threshold)

    print("\n==== KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST (BiGRU-ATTENTION OPTIMIZED) ====")
    print(f"Threshold sử dụng         : {optimal_threshold:.4f}")
    print(f"Accuracy                  : {test_acc*100:.2f}%")
    print(f"Recall (nhạy ca té ngã)   : {test_rec*100:.2f}%")
    print(f"F1-Score                  : {test_f1*100:.2f}%")
    print("=" * 60)

    os.makedirs("./models", exist_ok=True)
    np.save('./models/best_bigru_attn_config.npy', {**best_config, 'threshold': optimal_threshold})
    torch.save(best_model_state, './models/best_bigru_attn_fall_model.pth')
    print("[OK] Đã lưu model tại './models/best_bigru_attn_fall_model.pth'")

    os.makedirs("./plots", exist_ok=True)
    epochs_ran = len(history['train_loss'])

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_ran + 1), history['train_loss'], label='Train Loss', color='blue')
    plt.plot(range(1, epochs_ran + 1), history['val_loss'],   label='Val Loss',   color='orange', linestyle='--')
    plt.title('Loss History (BiGRU-Attention Optimized)')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs_ran + 1), history['train_acc'], label='Train Accuracy', color='green')
    plt.plot(range(1, epochs_ran + 1), history['val_acc'],   label='Val Accuracy',   color='red',  linestyle='--')
    plt.title('Accuracy History (BiGRU-Attention Optimized)')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig('./plots/bigru_attn_training_history.png', dpi=300); plt.close()

    cm = confusion_matrix(true_labels, pred_labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Oranges',
        xticklabels=['ADL (Bình thường)', 'Fall (Té ngã)'],
        yticklabels=['ADL (Bình thường)', 'Fall (Té ngã)']
    )
    plt.title(f'Normalized Confusion Matrix (BiGRU-Attention, threshold={optimal_threshold:.2f})')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('./plots/bigru_attn_normalized_confusion_matrix.png', dpi=300); plt.close()
    print("[OK] Toàn bộ pipeline đã được tối ưu hoàn tất.")

if __name__ == "__main__":
    main()

Đang sử dụng cấu hình: cuda

==== GIAI ĐOẠN 1: GRID SEARCH (BiGRU-Attention) ====
  H=32 L=1 LR=0.001 | Val F1: 83.8% | Val Recall: 86.7%
  H=32 L=1 LR=0.005 | Val F1: 84.8% | Val Recall: 86.4%
  H=32 L=2 LR=0.001 | Val F1: 85.9% | Val Recall: 86.4%
  H=32 L=2 LR=0.005 | Val F1: 83.4% | Val Recall: 84.0%
  H=64 L=1 LR=0.001 | Val F1: 87.6% | Val Recall: 85.1%
  H=64 L=1 LR=0.005 | Val F1: 81.1% | Val Recall: 79.3%
  H=64 L=2 LR=0.001 | Val F1: 89.2% | Val Recall: 91.0%
  H=64 L=2 LR=0.005 | Val F1: 85.2% | Val Recall: 82.9%

[*] Config tối ưu chọn theo F1: {'hidden_size': 64, 'num_layers': 2, 'lr': 0.001}

==== GIAI ĐOẠN 2: HUẤN LUYỆN CHI TIẾT (BiGRU-Attention) ====
Epoch [01/50] TrainLoss: 0.0460 TrainAcc: 80.1% | ValLoss: 0.0441 ValAcc: 80.7% ValF1: 68.1% ValRecall: 84.2%
  ==> Cải thiện! Toàn diện Val F1=68.1% — đã lưu trạng thái model.
Epoch [02/50] TrainLoss: 0.0397 TrainAcc: 81.4% | ValLoss: 0.0387 ValAcc: 82.7% ValF1: 69.6% ValRecall: 81.2%
  ==> Cải thiện! Toàn diện Val F1=69.6